# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nando135/flyrank_ml_1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The action queue uses the four K-Means archetypes as decision-support rather than automatic decisions. Cluster 1 receives the highest priority because it represents pages with relatively high visibility but weak CTR and engagement. Cluster 0 contains low-visibility, low-response pages and is lower priority. Cluster 3 contains pages with stronger engagement and should generally be monitored or protected. Cluster 2 contains extremely low-volume pages with unstable rates, so more data should be collected before acting.

In [3]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{hf_token}'
)
""")

query = """
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
"""

baseline_df = con.sql(query).df()

page_df = (
    baseline_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        sessions=("ga4_sessions", "sum"),
        engaged_sessions=("ga4_engaged_sessions", "sum")
    )
)

page_df["ctr"] = (
    page_df["clicks"] / page_df["impressions"]
).fillna(0)

page_df["engagement_rate"] = (
    page_df["engaged_sessions"] / page_df["sessions"]
).fillna(0)

feature_cols = [
    "impressions",
    "ctr",
    "sessions",
    "engagement_rate"
]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        page_df,
        groups=page_df["client_hash_id"]
    )
)

train_df = page_df.iloc[train_idx].copy()
val_df = page_df.iloc[val_idx].copy()

train_features = train_df[feature_cols].copy()
val_features = val_df[feature_cols].copy()

train_features["impressions"] = np.log1p(
    train_features["impressions"]
)

train_features["sessions"] = np.log1p(
    train_features["sessions"]
)

val_features["impressions"] = np.log1p(
    val_features["impressions"]
)

val_features["sessions"] = np.log1p(
    val_features["sessions"]
)

scaler = StandardScaler()

X_train = scaler.fit_transform(train_features)
X_val = scaler.transform(val_features)

model = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

train_df["cluster"] = model.fit_predict(X_train)
val_df["cluster"] = model.predict(X_val)

page_df = pd.concat(
    [train_df, val_df],
    ignore_index=True
)

print("Pages:", len(page_df))

print("\nCombined cluster counts:")
print(page_df["cluster"].value_counts().sort_index())

print("\nTraining cluster counts:")
print(train_df["cluster"].value_counts().sort_index())

print("\nValidation cluster counts:")
print(val_df["cluster"].value_counts().sort_index())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages: 63856

Combined cluster counts:
cluster
0    37657
1    23802
2     1158
3     1239
Name: count, dtype: int64

Training cluster counts:
cluster
0    34153
1    23398
2     1008
3     1120
Name: count, dtype: int64

Validation cluster counts:
cluster
0    3504
1     404
2     150
3     119
Name: count, dtype: int64


In [5]:
cluster_summary = (
    page_df
    .groupby("cluster")
    .agg(
        n=("content_hash_id", "count"),
        median_impressions=("impressions", "median"),
        median_ctr=("ctr", "median"),
        median_sessions=("sessions", "median"),
        median_engagement_rate=("engagement_rate", "median")
    )
    .reset_index()
)

display(cluster_summary)

,cluster,n,median_impressions,median_ctr,median_sessions,median_engagement_rate
0,0,37657,31.0,0.000000,2.0,0.000000
1,1,23802,1214.5,0.004265,23.0,0.000000
2,2,1158,1.0,1.000000,1.0,0.000000
3,3,1239,21.0,0.015385,2.0,0.666667


In [6]:
train_cluster_summary = (
    train_df
    .groupby("cluster")
    .agg(
        n=("content_hash_id", "count"),
        median_impressions=("impressions", "median"),
        median_ctr=("ctr", "median"),
        median_sessions=("sessions", "median"),
        median_engagement_rate=("engagement_rate", "median")
    )
    .reset_index()
)

print("Training")
display(train_cluster_summary)

val_cluster_summary = (
    val_df
    .groupby("cluster")
    .agg(
        n=("content_hash_id", "count"),
        median_impressions=("impressions", "median"),
        median_ctr=("ctr", "median"),
        median_sessions=("sessions", "median"),
        median_engagement_rate=("engagement_rate", "median")
    )
    .reset_index()
)

print("Validation")
display(val_cluster_summary)

Training


,cluster,n,median_impressions,median_ctr,median_sessions,median_engagement_rate
0,0,34153,33.0,0.000000,2.0,0.000000
1,1,23398,1222.0,0.004229,23.0,0.000000
2,2,1008,1.0,1.000000,1.0,0.000000
3,3,1120,22.5,0.014519,2.0,0.666667


Validation


,cluster,n,median_impressions,median_ctr,median_sessions,median_engagement_rate
0,0,3504,13.0,0.000000,1.0,0.000000
1,1,404,958.0,0.006306,10.0,0.000000
2,2,150,1.0,1.000000,1.0,0.000000
3,3,119,10.0,0.030303,2.0,0.666667


In [7]:
action_map = {
    0: {
        "action": "review_if_strategic",
        "reason_code": "low_visibility_low_response",
        "priority_level": 2
    },
    1: {
        "action": "review_content_and_search_snippet",
        "reason_code": "high_visibility_low_response",
        "priority_level": 1
    },
    2: {
        "action": "collect_more_data",
        "reason_code": "insufficient_volume_extreme_rate",
        "priority_level": 4
    },
    3: {
        "action": "protect_and_monitor",
        "reason_code": "engaged_low_volume",
        "priority_level": 3
    }
}

page_df["action"] = page_df["cluster"].map(
    lambda x: action_map[x]["action"]
)

page_df["reason_code"] = page_df["cluster"].map(
    lambda x: action_map[x]["reason_code"]
)

page_df["priority_level"] = page_df["cluster"].map(
    lambda x: action_map[x]["priority_level"]
)

page_df["within_cluster_priority"] = (
    page_df
    .groupby("cluster")["impressions"]
    .rank(method="first", ascending=False)
)

queue_df = (
    page_df
    .sort_values(
        ["priority_level", "within_cluster_priority"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

queue_df["queue_rank"] = queue_df.index + 1

display(
    queue_df[
        [
            "queue_rank",
            "client_hash_id",
            "content_hash_id",
            "cluster",
            "impressions",
            "ctr",
            "sessions",
            "engagement_rate",
            "action",
            "reason_code"
        ]
    ].head(20)
)

,queue_rank,client_hash_id,content_hash_id,cluster,impressions,ctr,sessions,engagement_rate,action,reason_code
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,1,617124,0.009185,2519,0.080985,review_content_and_search_snippet,high_visibility_low_response
1,2,client_e547b89c05043229,content_ec2e0346994fb5a5,1,245276,0.006034,719,0.111266,review_content_and_search_snippet,high_visibility_low_response
2,3,client_23a62021009f63c4,content_e8a52cf3d5988c07,1,244931,0.002731,891,0.115600,review_content_and_search_snippet,high_visibility_low_response
3,4,client_e547b89c05043229,content_0e03de7680314cd5,1,221310,0.003253,455,0.127473,review_content_and_search_snippet,high_visibility_low_response
4,5,client_23a62021009f63c4,content_36e53e9c707674fc,1,194579,0.001244,2603,0.013446,review_content_and_search_snippet,high_visibility_low_response
5,6,client_e547b89c05043229,content_4ffe18112a5642e3,1,186983,0.003134,347,0.164265,review_content_and_search_snippet,high_visibility_low_response
6,7,client_e547b89c05043229,content_8d7d99f109e19aa2,1,181942,0.001572,159,0.100629,review_content_and_search_snippet,high_visibility_low_response
7,8,client_23a62021009f63c4,content_44f34c0a90047651,1,168160,0.000089,37,0.027027,review_content_and_search_snippet,high_visibility_low_response
8,9,client_23a62021009f63c4,content_3df3f32f3fd58dea,1,137891,0.001407,347,0.069164,review_content_and_search_snippet,high_visibility_low_response
9,10,client_23a62021009f63c4,content_66288edeb93b7c4f,1,137878,0.005672,964,0.086100,review_content_and_search_snippet,high_visibility_low_response


The queue prioritizes Cluster 1 first because it contains higher-visibility pages with weak response, followed by lower-visibility weak-response pages. High-engagement pages are monitored rather than aggressively changed, while very low-volume pages with extreme rates are deprioritized until more evidence is available. Within each cluster, pages with more impressions are ranked first because they represent more observed visibility and potentially greater review value.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This playbook is intended to help a human reviewer prioritize which content pages should be reviewed first. It uses the validated K-Means clusters, visibility, CTR, sessions, and engagement patterns as decision-support signals rather than automatic decisions. The queue is most useful for March 2026 data with the same feature definitions and preprocessing used in the model.
The recommendations should not be treated as universal content-quality rules. Cluster membership is descriptive, and pages with very small impression or session counts can produce unstable CTR or engagement rates. The playbook should therefore be used to guide review, not to automatically rewrite, remove, or publish content.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Before acting on any recommendation, a human reviewer should check whether the page has enough data, whether the issue is still current, and whether there is a clear business or editorial reason to change the page. The reviewer should also inspect the actual content, search intent, and recent performance rather than relying on the cluster label alone.
The system should not automatically rewrite, delete, publish, redirect, or deindex content. It should also avoid acting automatically on pages with very small impression or session counts, because their CTR and engagement rates can be unstable. Cluster assignments and reason codes should be treated as review cues, not final judgments.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The recommendations should be reviewed when the underlying traffic and engagement patterns change materially from the March 2026 reference data. Retraining should be considered if cluster sizes shift substantially, cluster profiles such as median impressions, CTR, sessions, or engagement rate change noticeably, or the validation silhouette score drops meaningfully from the current value of about 0.466. A retrain may also be needed when new clients or new types of content behave differently from the existing clusters. These triggers are monitoring signals rather than automatic retraining rules, so a human should first confirm that the change reflects a real data shift rather than temporary noise or missing data.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("work/outputs", exist_ok=True)

export_cols = [
    "queue_rank",
    "client_hash_id",
    "content_hash_id",
    "cluster",
    "impressions",
    "ctr",
    "sessions",
    "engagement_rate",
    "priority_level",
    "action",
    "reason_code"
]

queue_df[export_cols].to_csv(
    "work/outputs/w07_ranked_action_queue.csv",
    index=False
)

print("Saved: work/outputs/w07_ranked_action_queue.csv")
print("Rows exported:", len(queue_df))

Saved: work/outputs/w07_ranked_action_queue.csv
Rows exported: 63856


In [12]:
import json

metrics = {
    "reference_month": "2026-03",
    "total_pages": int(len(page_df)),
    "clusters": 4,
    "validation_silhouette": 0.465995,
    "cluster_counts": {
        str(k): int(v)
        for k, v in page_df["cluster"].value_counts().sort_index().items()
    }
}

with open(
    "work/outputs/w07_playbook_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=2)

print("Saved: work/outputs/w07_playbook_metrics.json")

Saved: work/outputs/w07_playbook_metrics.json


In [13]:
import json

with open("work/outputs/w07_playbook_metrics.json", "r") as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))

{
  "reference_month": "2026-03",
  "total_pages": 63856,
  "clusters": 4,
  "validation_silhouette": 0.465995,
  "cluster_counts": {
    "0": 37657,
    "1": 23802,
    "2": 1158,
    "3": 1239
  }
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.